In [1]:
# this notebook will normalize(scale)/ clean the data
# while keeping key level as feature
# revert file : Cmd+Shift+P 
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

symbol = "BTCUSDT"
tf = "5m"

# 65 MB with almost 50 cols, LOL
folder_path = find_project_root() / "data" / "mlData" 
src_path = folder_path / "augmented" / f"{symbol}-{tf}-vX-augmented.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)

# extra col : experimenting data
# df["DELTA_2"] = df["DELTA_1"].rolling(2).sum()

print(df.columns)
df.head()

Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5',
       'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_5', 'RSI_14',
       'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5',
       'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,RSI_5,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609187400000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,-200.274933,0.332693,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1609187700000,NaN,0.000999,0.000999,0.000999,0.248526,0.000999,-97.010040,0.408053,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1609188000000,NaN,0.001603,0.001603,0.001603,0.398567,0.000603,-57.147125,0.410873,-354.432098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1609188300000,NaN,0.000301,0.000301,0.000301,0.074776,-0.001300,-36.593033,0.427996,-190.750198,...,73.868942,90.650436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1609188600000,NaN,0.003239,0.004242,0.004242,0.806364,0.003940,20.069031,0.541244,-73.671127,...,86.860428,92.834373,NaN,88.131424,156.969452,NaN,NaN,NaN,NaN,NaN


# Analyse Data
- Drop warm-up rows
- check Infinity, NaN, None
- verify data's quality. Non-staionary, local memory, etc.

In [2]:
# removing warmup nan and check label imbalance
import numpy as np
# Replace inf with NaN 
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# list feature columns
# Drop RSI_5 from VIF test
# Drop '15STR_confirmed','45STR_confirmed','barsSince15STR','barsSince45STR' from spearman test
group_I = ['ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1']
group_J = ['ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO']
group_K = ['RSI_5', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5']
group_L = ['DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV']
group_M = ['DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS']
# group_N = [
#     '15STR_confirmed', '45STR_confirmed',
#     'barsSince15STR',  'barsSince45STR',
#     'DIST_HIGH_15STR', 'DIST_LOW_15STR',
#     'DIST_HIGH_45STR', 'DIST_LOW_45STR',
#     'RANGE_15STR',     'RANGE_45STR',
# ]

feature_cols = [*group_I, *group_J, *group_K, *group_L, *group_M]

# make sure that we drop only warm-up rows
df_clean = df.dropna(subset=feature_cols).reset_index(drop=True)

dropped = len(df) - len(df_clean)
print(f"Rows before : {len(df):,}")
print(f"Rows dropped: {dropped:,}  (warmup NaN from rolling windows)")
print(f"Rows after cleaned  : {len(df_clean):,}")

Rows before : 546,454
Rows dropped: 45  (warmup NaN from rolling windows)
Rows after cleaned  : 546,409


In [3]:
# Trim start-up and tail NaN rows from label (only contiguous edges)
# also make sure there was not supposed to had any NaN left in any rows or columns

label = df_clean['label']

# Find first and last valid label index
first_valid_idx = label.first_valid_index()
last_valid_idx  = label.last_valid_index()

df_clean = df_clean.loc[first_valid_idx:last_valid_idx].reset_index(drop=True)

print(f"Label NaN trimmed — head up to index {first_valid_idx}, tail after index {last_valid_idx}")
print(f"Rows after trim: {len(df_clean):,}")

# Chronological order must still hold
assert df_clean['timestamp'].is_monotonic_increasing, "Timestamp order broken after trim!"

# Assert no NaN remains anywhere — only contiguous edge NaN in label were expected
remaining_nan = df_clean.isnull().sum()
remaining_nan = remaining_nan[remaining_nan > 0]
assert len(remaining_nan) == 0, f"Unexpected NaN still present:\n{remaining_nan.to_string()}"

print("Chronological order: OK")
print("All columns clean: no NaN remaining.")

# train all 3 labels
trainable = df_clean
n_up   = (trainable['label'] ==  1).sum()
n_down = (trainable['label'] == -1).sum()
total  = len(trainable)

print(f"\nLabel balance (trainable only):")
print(f"  Up   ( 1) : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1) : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout   : {(df_clean['label'] == 0).sum():,}  ({(df_clean['label'] == 0).sum()/total*100:.1f})")
print(f"  Total     : {total:,}  ({total / len(df_clean) * 100:.1f}% of clean rows)")


Label NaN trimmed — head up to index 28, tail after index 546408
Rows after trim: 546,381
Chronological order: OK
All columns clean: no NaN remaining.

Label balance (trainable only):
  Up   ( 1) : 174,145  (31.9%)
  Down (-1) : 179,668  (32.9%)
  Timeout   : 192,568  (35.2)
  Total     : 546,381  (100.0% of clean rows)


In [4]:
# Nan and Infinity re-check
# import numpy as np

# 1. Feature columns must be clean (no NaN/Inf from warm-up)
assert df_clean[feature_cols].isnull().sum().sum() == 0, "NaN leaked through on feature columns!"
assert not np.isinf(df_clean[feature_cols].values).any(), "Inf leaked through on feature columns!"

# 2. Show any remaining NaN per column (expected only in 'label' from source data)
nan_counts = df_clean.isnull().sum()
nan_remaining = nan_counts[nan_counts > 0]
if len(nan_remaining) > 0:
    print("NaN remaining (source data NaN, NOT warm-up):")
    print(nan_remaining.to_string())
else:
    print("No NaN remaining in any column.")

# 3. Verify chronological order is preserved
assert df_clean['timestamp'].is_monotonic_increasing, "Timestamp order broken!"
print(f"\nChronological order: OK (timestamps monotonically increasing)")
print(f"Feature columns: clean (no NaN, no Inf)")


No NaN remaining in any column.

Chronological order: OK (timestamps monotonically increasing)
Feature columns: clean (no NaN, no Inf)


In [5]:
print(df_clean.shape)
print(df_clean.columns)
del df

(546381, 27)
Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5',
       'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_5', 'RSI_14',
       'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5',
       'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


# Test : Verify Data Quality
do them on df_clean
see [Data Quality](../../../../../journal/trainData/dataQuality.md)
- Are all features stationary
- Does it preserve local memory?
- Are features redundant with each other?
- Does it actually relate to the label?

# Skip to save Data if all had passed

# Note
- all passed, drop RSI_5 because VIF test exceed 30 + High MI correlation

In [6]:
# Group I — Rate of Change (passed)
# Group L — Order Flow (passed)
# Group J — ATR Volatility Ratio
# Group K — RSI & Momentum Oscillators
# Group M — Distance to Swing High / Low
# Group N — ICT Pivot Distance (15STR + 45STR)

GLOBAL_FEATURES = [
    # Group I
    "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",
    # Group J
    "ATR_RATIO", "ATR_NORM_ROC", "RANGE_RATIO",
    # Group K
    "RSI_14", "RSI_SLOPE", "CCI_5",
    # Group L
    "VOL_SPIKE", "DELTA_DIV",
    # Group M 
]

ROLLING_FEATURES = [
    "DELTA_1", "DELTA_3", "BUY_RATIO",
    "ATR_5", "ATR_14",
    "STOCH_K",
    "DIST_HIGH_5", "DIST_HIGH_10", "RANGE_POS",
    "DIST_LOW_5", "DIST_LOW_10",
]

In [ ]:
# Test1 : stationary test
# ADF  rejects unit root     → not non-stationary
# KPSS rejects stationarity  → not trend-stationary
from statsmodels.tsa.stattools import adfuller, kpss
# import pandas as pd
# import numpy as np

# Full sample
def stationarity_report1(series: pd.Series, name: str) -> dict:
    """
    ADF  null: series HAS a unit root (non-stationary)
         want: reject null → p < 0.05 → stationary

    KPSS null: series IS stationary
         want: fail to reject → p > 0.05 → stationary
    """
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series.dropna(), autolag="AIC")
    kpss_stat, kpss_p, _, kpss_crit    = kpss(series.dropna(), regression="c", nlags="auto")

    verdict = "PASS" if (adf_p < 0.05 and kpss_p > 0.05) else \
              "WARN" if (adf_p < 0.05 or  kpss_p > 0.05) else "FAIL"

    return {
        "feature":   name,
        "adf_p":     round(adf_p,   4),
        "kpss_p":    round(kpss_p,  4),
        "adf_stat":  round(adf_stat, 3),
        "kpss_stat": round(kpss_stat,3),
        "verdict":   verdict,
    }

# sub sample
def stationarity_report2(series: pd.Series, name: str, max_n: int = 10_000) -> dict:
    s = series.dropna()
    if len(s) > max_n:
        idx = np.linspace(0, len(s) - 1, max_n, dtype=int)
        s = s.iloc[idx]
    
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(s, autolag="AIC")
    kpss_stat, kpss_p, _, kpss_crit    = kpss(s, regression="c", nlags="auto")

    verdict = "PASS" if (adf_p < 0.05 and kpss_p > 0.05) else \
              "WARN" if (adf_p < 0.05 or  kpss_p > 0.05) else "FAIL"

    return {
        "feature":   name,
        "adf_p":     round(adf_p,   4),
        "kpss_p":    round(kpss_p,  4),
        "adf_stat":  round(adf_stat, 3),
        "kpss_stat": round(kpss_stat, 3),
        "verdict":   verdict,
    }

# one-by-one test
# stationarity_report(df["barsSince15STR"],"barsSince15STR")
# stationarity_report(df["barsSince45STR"],"barsSince45STR")

# loop test
features_to_test = {f: df_clean[f] for f in GLOBAL_FEATURES}
# sub sample test
results = [stationarity_report2(series, name) for name, series in features_to_test.items()]
pd.DataFrame(results)

In [ ]:
# Test1-2 : Stationary on Rolling z-score
# Due to regime shift some feature has been passed to rolling-z-transform

# import numpy as np
# import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings("ignore")

# ── Constants ─────────────────────────────────────────────────────────
WINDOW      = 500
TRAIN_END   = int(546381 * 0.80)   # 437,104
SAMPLE_SIZE = 10_000               # subsample for speed — ADF/KPSS are O(n²)

# ROLLING_FEATURES = [
#     "DELTA_1", "DELTA_3", "BUY_RATIO",
#     "ATR_5", "ATR_14",
#     "STOCH_K",
#     "DIST_HIGH_5", "DIST_HIGH_10", "RANGE_POS"
# ]

# ── Apply rolling Z-score (same as pipeline) ──────────────────────────
def rolling_zscore(series, window):
    rm  = series.rolling(window).mean()
    rs  = series.rolling(window).std()
    return (series - rm) / rs

# ── ADF + KPSS on a single series ────────────────────────────────────
def test_stationarity(series, name, segment):
    s = series.dropna()
    if len(s) > SAMPLE_SIZE:
        # Systematic subsample — evenly spaced, not random, preserves temporal structure
        idx = np.linspace(0, len(s) - 1, SAMPLE_SIZE, dtype=int)
        s = s.iloc[idx]

    # ADF — H0: unit root (non-stationary). Reject H0 → stationary
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(s, autolag="AIC")
    adf_pass = adf_p < 0.05

    # KPSS — H0: stationary. Fail to reject H0 → stationary
    kpss_stat, kpss_p, _, kpss_crit = kpss(s, regression="c", nlags="auto")
    kpss_pass = kpss_p > 0.05

    both_pass = adf_pass and kpss_pass
    status    = "PASS ✓" if both_pass else ("ADF only" if adf_pass else ("KPSS only" if kpss_pass else "FAIL ✗"))

    return {
        "feature":   name,
        "segment":   segment,
        "adf_stat":  adf_stat,
        "adf_p":     adf_p,
        "adf_pass":  adf_pass,
        "kpss_stat": kpss_stat,
        "kpss_p":    kpss_p,
        "kpss_pass": kpss_pass,
        "status":    status,
    }

# ── Run tests ─────────────────────────────────────────────────────────
results = []

for feat in ROLLING_FEATURES:
    raw    = df_clean[feat]
    normed = rolling_zscore(raw, WINDOW).iloc[WINDOW:]   # drop warmup

    train  = normed.iloc[:TRAIN_END - WINDOW]
    full   = normed

    results.append(test_stationarity(train, feat, "train"))
    results.append(test_stationarity(full,  feat, "full"))

# ── Print results ─────────────────────────────────────────────────────
rdf = pd.DataFrame(results)

print(f"\nADF + KPSS — Rolling Z-score features (window={WINDOW})")
print(f"Subsample size: {SAMPLE_SIZE:,} bars  |  ADF: p<0.05 = stationary  |  KPSS: p>0.05 = stationary")
print("─" * 95)
print(f"{'Feature':<20} {'Segment':<8} {'ADF stat':>10} {'ADF p':>10} {'ADF':>6} {'KPSS stat':>10} {'KPSS p':>10} {'KPSS':>6} {'Result':>10}")
print("─" * 95)

for _, row in rdf.iterrows():
    print(
        f"{row['feature']:<20} {row['segment']:<8} "
        f"{row['adf_stat']:>10.4f} {row['adf_p']:>10.4f} {'✓' if row['adf_pass'] else '✗':>6} "
        f"{row['kpss_stat']:>10.4f} {row['kpss_p']:>10.4f} {'✓' if row['kpss_pass'] else '✗':>6} "
        f"{row['status']:>10}"
    )
    if row['segment'] == 'full':
        print("─" * 95)

# ── Summary ───────────────────────────────────────────────────────────
pass_both = rdf[rdf['status'] == 'PASS ✓']
fail      = rdf[rdf['status'] == 'FAIL ✗']

print(f"\nSummary: {len(pass_both)}/{len(rdf)} segments passed both ADF and KPSS")
if len(fail) > 0:
    print("Failures:")
    for _, row in fail.iterrows():
        print(f"  {row['feature']} ({row['segment']}) — ADF p={row['adf_p']:.4f}, KPSS p={row['kpss_p']:.4f}")


In [ ]:
# Group N

# feature	adf_p	kpss_p	adf_stat	kpss_stat	verdict
# 0	15STR_confirmed	0.0	0.0431	-61.503	0.494	WARN
# 1	45STR_confirmed	0.0	0.1000	-70.134	0.065	PASS
# 2	barsSince15STR	0.0	0.0100	-50.246	1.292	WARN
# 3	barsSince45STR	0.0	0.1000	-47.477	0.194	PASS
# 4	DIST_HIGH_15STR	0.0	0.0357	-49.844	0.526	WARN
# 5	DIST_LOW_15STR	0.0	0.1000	-49.247	0.183	PASS
# 6	DIST_HIGH_45STR	0.0	0.0100	-33.960	0.905	WARN
# 7	DIST_LOW_45STR	0.0	0.1000	-32.399	0.307	PASS
# 8	RANGE_15STR	0.0	0.1000	-61.728	0.127	PASS
# 9	RANGE_45STR	0.0	0.0929	-35.956	0.363	PASS



In [ ]:
# Test extra : Rolling - zscore preview
import matplotlib.pyplot as plt

total_rows = len(df_clean)

# ── Split boundaries (80/10/10 on 546,381 cleaned rows) ──────────────
TRAIN_END = int(total_rows * 0.80)   # 437,104
VAL_END   = int(total_rows * 0.90)   # 491,742
# Test: 491,742 → 546,381

# ROLLING_FEATURES = [
#     "DELTA_1", "DELTA_3", "BUY_RATIO",
#     "ATR_5", "ATR_14",
#     "STOCH_K",
#     "DIST_HIGH_5", "DIST_HIGH_10", "RANGE_POS"
# ]

window = 500

print(f"Split boundaries → Train: 0–{TRAIN_END} | Val: {TRAIN_END}–{VAL_END} | Test: {VAL_END}–546381")
print(f"{'Feature':<20} {'Train_early':>12} {'Train_mid':>12} {'Train_late':>12} {'Val':>12} {'Test':>12} {'Global_std':>12}")
print("─" * 92)

for feat in ROLLING_FEATURES:
    rs = df_clean[feat].rolling(window).std()

    train_early = rs.iloc[window:TRAIN_END//3].mean()
    train_mid   = rs.iloc[TRAIN_END//3 : 2*TRAIN_END//3].mean()
    train_late  = rs.iloc[2*TRAIN_END//3 : TRAIN_END].mean()
    val_mean    = rs.iloc[TRAIN_END:VAL_END].mean()
    test_mean   = rs.iloc[VAL_END:].mean()
    global_std  = rs.iloc[window:].std()

    print(f"{feat:<20} {train_early:>12.4f} {train_mid:>12.4f} {train_late:>12.4f} {val_mean:>12.4f} {test_mean:>12.4f} {global_std:>12.4f}")

# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(ROLLING_FEATURES):
    rs = df[feat].rolling(window).std()
    axes[i].plot(rs.values, linewidth=0.6, color="steelblue")
    axes[i].axvline(TRAIN_END, color="orange", linewidth=1.2, linestyle="--", label="train/val")
    axes[i].axvline(VAL_END,   color="red",    linewidth=1.2, linestyle="--", label="val/test")
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel("Bar index")
    axes[i].set_ylabel("Rolling std")
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle(f"Rolling Std (window={window}) — 9 Features across 80/10/10 Split", fontsize=12)
plt.tight_layout()
# plt.savefig("rolling_std_check.png", dpi=150)
plt.show()
# print("Saved: rolling_std_check.png")

# Stationary Test End
```
Rolling Z-score (9 features):
  DELTA_1, DELTA_3, BUY_RATIO
  ATR_5, ATR_14
  STOCH_K
  DIST_HIGH_5, DIST_HIGH_10, RANGE_POS

Global Z-score (16 features):
  Everything else

WARN but no action (confirmed artefacts):
  DELTA_DIV   — binary ACF clustering
  ATR_NORM_ROC, RSI_5, RSI_14  — autocorrelation, stable rolling mean
  DIST_LOW_5, DIST_LOW_10      — rolling min clustering, stable rolling mean
```

In [ ]:
# Test2 : local memory test
# Sample Size Inflates Ljung-Box. Use another method

from statsmodels.tsa.stattools import acf
# import pandas as pd
# import numpy as np

def acf_magnitude_report(series: pd.Series, name: str, max_lag: int = 20):
    """
    Look at actual ACF coefficients, not just significance flags.
    Thresholds:
        |r| > 0.10  → meaningful memory
        |r| > 0.05  → weak but present
        |r| < 0.02  → negligible (likely noise even if p < 0.05)
    """
    acf_vals, confint = acf(series.dropna(), nlags=max_lag,
                            alpha=0.05, fft=True)

    # skip lag 0 (always 1.0)
    lags      = range(1, max_lag + 1)
    acf_lags  = acf_vals[1:]
    ci_lower  = confint[1:, 0] - acf_vals[1:]
    ci_upper  = confint[1:, 1] - acf_vals[1:]

    meaningful = [lag for lag, r in zip(lags, acf_lags) if abs(r) > 0.05]
    strong     = [lag for lag, r in zip(lags, acf_lags) if abs(r) > 0.10]

    print(f"\n{name}")
    print(f"  ACF at lag 1:  {acf_lags[0]:+.4f}")
    print(f"  ACF at lag 3:  {acf_lags[2]:+.4f}")
    print(f"  ACF at lag 5:  {acf_lags[4]:+.4f}")
    print(f"  ACF at lag 10: {acf_lags[9]:+.4f}")
    print(f"  ACF at lag 20: {acf_lags[19]:+.4f}")
    print(f"  Lags with |r| > 0.05 (weak):     {meaningful}")
    print(f"  Lags with |r| > 0.10 (meaningful): {strong}")

    return pd.DataFrame({
        "lag": list(lags),
        "acf": acf_lags.round(4),
    })

features_to_test = [*GLOBAL_FEATURES, *ROLLING_FEATURES]

# run on all features
for name in features_to_test:
    acf_magnitude_report(df_clean[name], name)

# group - N 
# a confirmed swing is a point event, not a persistent state.
# extra check: till lag 50
# if some params don't decay to zero

# Local memory test End
```
All 25 features pass local memory check.

25/25 carry meaningful ACF within LSTM receptive field (lags 1–14)
 3/25 extend beyond lag 20 (ATR_5, ATR_14, RSI_14) — structural, acceptable, handled by 500 rolling standardization
 0/25 require any pipeline change based on ACF results
```

In [ ]:
# Test 3 : Cross correlation test
# Are features redundant with each other ?
import matplotlib.pyplot as plt
import seaborn as sns


def correlation_report(df_features: pd.DataFrame, method: str = "spearman"):
    """
    Spearman preferred over Pearson — does not assume linearity.
    Flag pairs with |r| > 0.85 as redundant.
    """
    corr = df_features.corr(method=method)

    # find high-correlation pairs
    pairs = []
    cols  = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            r = corr.iloc[i, j]
            if abs(r) > 0.85:
                pairs.append({"feature_a": cols[i],
                              "feature_b": cols[j],
                              "spearman_r": round(r, 3)})

    if pairs:
        print("High correlation pairs (|r| > 0.85) — consider dropping one:")
        print(pd.DataFrame(pairs).to_string(index=False))
    else:
        print("No redundant pairs found.")

    # heatmap - plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1)
    plt.title(f"Feature Correlation ({method.title()})")
    plt.tight_layout()
    plt.savefig("feature_correlation.png", dpi=150)
    plt.show()

    return corr

features_to_test = [*GLOBAL_FEATURES, *ROLLING_FEATURES]
feature_df = pd.DataFrame({name: df_clean[name] for name in features_to_test})
correlation_report(feature_df)

In [ ]:
# Test 3 extra - VIF : variance_inflation_factor
# Further Verification, if we really had to drop a column
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
# import numpy as np

def vif_report(X: np.ndarray, feature_names: list):
    vif = pd.DataFrame({
        "feature": feature_names,
        "VIF": [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    }).sort_values("VIF", ascending=False)
    print(vif.to_string(index=False))
    return vif

# Drop RSI_5

# feature_names = [
#     "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",                      # Group I
#     "DELTA_1", "DELTA_3", "BUY_RATIO", "VOL_SPIKE", "DELTA_DIV",           # Group L
#     "ATR_5", "ATR_14", "ATR_RATIO", "ATR_NORM_ROC", "RANGE_RATIO",         # Group J
#     "RSI_14", "RSI_SLOPE", "STOCH_K", "CCI_5",                             # Group K
#     "DIST_HIGH_5", "DIST_LOW_5", "DIST_HIGH_10", "DIST_LOW_10", "RANGE_POS", # Group M
#     "15STR_confirmed", "45STR_confirmed",                                   # Group N (ternary)
#     "barsSince15STR", "barsSince45STR",                                     # Group N (log1p)
#     "DIST_HIGH_15STR", "DIST_LOW_15STR",                                    # Group N
#     "DIST_HIGH_45STR", "DIST_LOW_45STR",                                    # Group N
#     "RANGE_15STR", "RANGE_45STR",                                           # Group N
# ]

features_to_test = [*GLOBAL_FEATURES, *ROLLING_FEATURES]

df_vif = df_clean.copy()

X_raw = df_vif[features_to_test].dropna().values

# Z-score first — VIF on raw unscaled features is misleading
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# run it
# low VIF. = good new orthogonal information
vif_result = vif_report(X_scaled, features_to_test)

# Feature Correlation test End
```
RSI_5 is the only feature above the drop threshold of 30. It is almost entirely explained by the combination of STOCH_K, RANGE_POS, DIST_LOW_10, and RSI_SLOPE — all of which encode the same "price position within recent range" information.
```

In [ ]:
# Test 4 Verify by tweaking k-NN with
# applied 500 rolling window and
# applied Z-score standardization

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
# import pandas as pd
# import numpy as np

# ── feature lists ─────────────────────────────────────────────────────────────
GLOBAL_FEATURES = [
    # Group I
    "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",
    # Group J
    "ATR_RATIO", "ATR_NORM_ROC", "RANGE_RATIO",
    # Group K
    "RSI_14", "RSI_SLOPE", "CCI_5",
    # Group L
    "VOL_SPIKE", "DELTA_DIV",
    # Group M 
]

ROLLING_FEATURES = [
    "DELTA_1", "DELTA_3", "BUY_RATIO",
    "ATR_5", "ATR_14",
    "STOCH_K",
    "DIST_HIGH_5", "DIST_HIGH_10", "RANGE_POS",
    "DIST_LOW_5", "DIST_LOW_10",
]


ALL_FEATURES = [*GLOBAL_FEATURES, *ROLLING_FEATURES]   # 24 features

ROLLING_WINDOW = 500

# ── Pre-process: log1p on barsSince before any scaling ───────────────────────
df_mi = df_clean.copy()

# ── Step 1: rolling Z-score ───────────────────────────────────────────────────
for feat in ROLLING_FEATURES:
    roll_mean = df_mi[feat].rolling(ROLLING_WINDOW).mean()
    roll_std  = df_mi[feat].rolling(ROLLING_WINDOW).std().replace(0, np.nan)
    df_mi[feat] = (df_mi[feat] - roll_mean) / roll_std

# ── Step 2: drop warmup rows ──────────────────────────────────────────────────
df_mi = df_mi.iloc[ROLLING_WINDOW:].dropna(subset=ALL_FEATURES + ["label"]).reset_index(drop=True)

# ── Step 3: global Z-score on everything else ─────────────────────────────────
X_raw = df_mi[ALL_FEATURES].values
y     = df_mi["label"].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# ── Step 4: winsorise ─────────────────────────────────────────────────────────
X_scaled = np.clip(X_scaled, -3, 3)

# ── Step 5: MI ────────────────────────────────────────────────────────────────
mi_k3  = mutual_info_classif(X_scaled, y, n_neighbors=3,  random_state=42)
mi_k10 = mutual_info_classif(X_scaled, y, n_neighbors=10, random_state=42)

report = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mi_k3":   mi_k3.round(4),
    "mi_k10":  mi_k10.round(4),
}).sort_values("mi_k3", ascending=False)

print(report.to_string(index=False))

# Mutual Information test End

In [ ]:
# Final test 5 : spearman on feature that has low MI
from scipy.stats import spearmanr

for feat in ALL_FEATURES:
    r, p = spearmanr(df_mi[feat], df_mi["label"])
    print(f"{feat:20s}  r={r:.4f}  p={p:.4f}")


# Export cleaned data

In [7]:
# drop columns concluded from test result
drop_col = ['BUY_RATIO','RSI_5']
df_clean.drop(columns=drop_col,inplace=True)
print(df_clean.columns)
df_clean.head()

Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5', 'ATR_14',
       'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE',
       'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10',
       'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,DELTA_3,VOL_SPIKE,...,RANGE_RATIO,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1609199700000,-1.0,-0.000751,-0.002339,0.005548,-0.189374,-0.002075,-11.871964,-66.426086,0.838335,...,0.930778,60.994850,-3.512627,37.723309,-81.744637,0.838698,0.508031,0.838698,1.485528,0.639149
1,1609200000000,-1.0,-0.004029,-0.002143,0.001350,-1.014461,-0.001890,-11.295532,-62.670853,1.539922,...,2.090774,56.747784,-9.465221,44.256199,-133.719208,1.444698,1.146976,1.444698,1.146976,0.442562
2,1609200300000,-1.0,-0.002666,-0.001344,-0.000056,-0.677750,0.001299,-2.558212,-25.725708,0.796544,...,0.804380,58.864796,-7.180096,58.700855,-44.470379,1.074053,1.526613,1.074053,1.526613,0.587009
3,1609200600000,-1.0,-0.001884,-0.004023,-0.000927,-0.479830,-0.001292,-19.458145,-33.311890,0.865680,...,0.814543,55.929337,-5.065514,44.318054,-79.413910,1.509120,1.201130,1.509120,1.201130,0.443181
4,1609200900000,-1.0,-0.002710,-0.006660,-0.005351,-0.679541,-0.002715,-25.977829,-47.994186,1.204642,...,1.621373,50.264042,-6.483742,30.484550,-156.113785,1.994144,0.874490,2.136262,0.874490,0.290456


In [8]:
# save the current data in ./data/mlData/clean-v3
# save to JSONL for training
out_path = folder_path / "clean" / f"{symbol}-{tf}-vX-cleaned.jsonl"

df_clean.to_json(out_path, orient="records", lines=True)
print(f"final shape : {df_clean.shape}")
print(f"Saved {len(df_clean)} rows to {out_path}")


final shape : (546381, 25)
Saved 546381 rows to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-candle-science/data/mlData/clean/BTCUSDT-5m-vX-cleaned.jsonl
